In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import pystormtracker as pst
from pathlib import Path
import cartopy.crs as ccrs

import sys
import os

module_dir = (Path.cwd() / ".." / '..'/"algorithms"/"track_style_alg").resolve()
sys.path.insert(0,str(module_dir))
import validation as val
import processing as pro
import helpers
from configs import *

import importlib
importlib.reload(val)  # pick up edits in validation.py without restarting the kernel
importlib.reload(pro)  # pick up edits in processing.py without restarting the kernel

import importlib


In [2]:
TIME_RANGE = ('1998-12-01 00:00:00', '1999-02-28 18:00:00')  # time range for the validation tracks

In [3]:
ds = xr.open_dataset('/Users/asgerthormann/Documents/uni/master/master_code/master_thesis/reanalysis_data/ERA5/ERA5_pressure_levels/19981999_DJF_globe_fejer.nc')
ds = ds.rename({'lon': 'longitude', 'lat': 'latitude'})
ds

<xarray.Dataset> Size: 332MB
Dimensions:         (valid_time: 720, pressure_level: 1, latitude: 240,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 6kB 1998-01-01 ... 1999-12-31...
  * pressure_level  (pressure_level) float64 8B 850.0
  * latitude        (latitude) float64 2kB -89.62 -88.88 -88.12 ... 88.88 89.62
  * longitude       (longitude) float64 4kB 0.0 0.75 1.5 ... 357.8 358.5 359.2
Data variables:
    vo              (valid_time, pressure_level, latitude, longitude) float32 332MB ...
Attributes:
    CDI:                     Climate Data Interface version 2.5.0 (https://mp...
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    history:                 Fri Oct 30 16:25:37 2026: cdo remapcon,r480x240 ...
    CDO:                     Climate Data Operators version 2.5.0 (https://mp...

In [4]:
ds = pro.prepare_tracking_field(ds, variable='vo',level =850, domain=None)
ds = ds.sel(valid_time=slice(TIME_RANGE[0], TIME_RANGE[1]))
ds

<xarray.Dataset> Size: 166MB
Dimensions:     (valid_time: 360, latitude: 240, longitude: 480)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 1998-12-01 ... 1999-02-28T18:...
  * latitude    (latitude) float64 2kB -89.62 -88.88 -88.12 ... 88.88 89.62
  * longitude   (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    vo          (valid_time, latitude, longitude) float32 166MB ...

In [5]:
tracks = pro.hodges_tracker(input_file_path=ds, output_file_name='19981999_DJF_vo_lmin_6_coarse', track_variable='vo', min_track_points=3, taper_points=0, lmin=6,lmax=42, spectral_taper=0.1, min_object_grid_points=1,
                    object_threshold=1e-5, exclude_boundary_extrema=False)

In [6]:
# POSTPROCESS_CONFIG = dict(region=REGION,
#                           domain=DOMAIN,
#                           time_range=TIME_RANGE,
#                           region_criterion="passes",
#                           min_points_in_region=1,
#                           rsplice_min_points=8,
#                           rsplice_max_points=None,
#                           rsplice_distance_degrees=10.0,
#                           rsplice_distance_mode="endpoint",
#                           rsplice_cadence=np.timedelta64(6, "h"),
#                           min_peak=None,
#                           max_frac_at_edge=0.5,
#                           edge_buffer_deg=2.5, drop_time_truncated=False,
# )

In [7]:
tracks, summary, points = pro.postprocess_tracks(tracks, **POSTPROCESS_CONFIG)

In [8]:
report = pro.filter_report(summary)

raw tracks: 3745, kept: 121
